In [10]:

import os
import json
from pathlib import Path
from typing import List, Tuple
import boto3
from tqdm import tqdm

# Initialize Textract client
textract_client = boto3.client('textract')

def get_image_files(folder_path: str = '.') -> List[Path]:
    """
    Get all jpg and png files from the specified folder.
    
    Args:
        folder_path: Path to the folder to scan (default: current directory)
    
    Returns:
        List of Path objects for image files
    """
    valid_extensions = {'.jpg', '.jpeg', '.png'}
    image_files = [
        Path(file) 
        for file in os.listdir(folder_path)
        if Path(file).suffix.lower() in valid_extensions
    ]
    return sorted(image_files)


def should_skip_file(image_path: Path) -> bool:
    """
    Check if a txt file already exists for the given image.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        True if txt file exists (should skip), False otherwise
    """
    txt_path = image_path.with_suffix('.txt')
    return txt_path.exists()


def extract_text_from_image(image_path: Path) -> str:
    """
    Extract text from an image using AWS Textract DetectDocumentText API.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        Extracted text from the image
    
    Raises:
        Exception: If Textract API call fails
    """
    with open(image_path, 'rb') as image_file:
        image_data = image_file.read()
    
    response = textract_client.detect_document_text(Document={'Bytes': image_data})
    
    # Extract and concatenate all text blocks
    extracted_text = '\n'.join(
        [block['Text'] for block in response['Blocks'] if block['BlockType'] == 'LINE']
    )
    
    return extracted_text


def save_extracted_text(image_path: Path, extracted_text: str) -> Path:
    """
    Save extracted text to a .txt file with the same name as the image.
    
    Args:
        image_path: Path to the original image file
        extracted_text: The extracted text to save
    
    Returns:
        Path to the saved txt file
    """
    txt_path = image_path.with_suffix('.txt')
    
    with open(txt_path, 'w', encoding='utf-8') as txt_file:
        txt_file.write(extracted_text)
    
    return txt_path


def process_single_image(image_path: str) -> bool:
    """
    Process a single image file.
    Extracts text using Textract and saves to a .txt file.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        True if processed, False if skipped
    """
    image_path = Path(image_path)
    
    if not image_path.exists():
        print(f"✗ Error: File not found: {image_path}")
        return False
    
    # Skip if txt file already exists
    if should_skip_file(image_path):
        print(f"⊘ Skipped: {image_path.name} (txt already exists)")
        return False
    
    try:
        print(f"Processing: {image_path.name}...")
        
        # Extract text from image
        extracted_text = extract_text_from_image(image_path)
        
        # Save to txt file
        txt_path = save_extracted_text(image_path, extracted_text)
        
        print(f"✓ Success: Text saved to {txt_path.name}")
        
        return True
        
    except Exception as e:
        print(f"✗ Error processing {image_path.name}: {str(e)}")
        return False


# Run the processing
process_single_image('Employment_verification.png')

Processing: Employment_verification.png...
✓ Success: Text saved to Employment_verification.txt


True